## Statistical Annotations: `geomBracket()` and `geomBracketDodge()`

`geomBracket()` is a general-purpose layer for annotating ranges. It is particularly effective for adding significance bars (p-values) to categorical plots.

`geomBracketDodge()` is a specialized layer designed for annotating ranges between dodged positions (usually dodged groups within a category).

> **Note:** `geomBracket` does not compute statistics internally. It is a visualization tool that renders the results of your analysis. In this demo, we use the Mann–Whitney U test to calculate p-values before passing them to the plot.

In [2]:
%useLatestDescriptors
%use dataframe
%use lets-plot

In [3]:
LetsPlot.getInfo()

Lets-Plot Kotlin API v.4.12.2-SNAPSHOT. Frontend: Notebook with dynamically loaded JS. Lets-Plot JS v.4.9.0.
Outputs: Web (HTML+JS), Kotlin Notebook (Swing), Static SVG (hidden)

In [4]:
// Mann-Whitney U test (two-sided) using normal approximation
fun mannWhitneyPValue(x: List<Double>, y: List<Double>): Double {
    val nx = x.size
    val ny = y.size
    val combined = (x.map { it to 0 } + y.map { it to 1 }).sortedBy { it.first }
    // Assign ranks (average for ties)
    val ranks = DoubleArray(nx + ny)
    var i = 0
    while (i < combined.size) {
        var j = i
        while (j < combined.size - 1 && combined[j].first == combined[j + 1].first) j++
        val avgRank = (i + j + 2).toDouble() / 2.0  // 1-based average rank
        for (k in i..j) ranks[k] = avgRank
        i = j + 1
    }
    val r1 = combined.indices.filter { combined[it].second == 0 }.sumOf { ranks[it] }
    val u1 = r1 - nx.toLong() * (nx + 1) / 2.0
    val u = minOf(u1, nx.toDouble() * ny - u1)
    val mean = nx.toDouble() * ny / 2.0
    val std = kotlin.math.sqrt(nx.toDouble() * ny * (nx + ny + 1) / 12.0)
    val z = (u - mean) / std
    // Two-sided p-value using complementary error function approximation
    fun erfc(x: Double): Double {
        val t = 1.0 / (1.0 + 0.3275911 * kotlin.math.abs(x))
        val poly = t * (0.254829592 + t * (-0.284496736 + t * (1.421413741 + t * (-1.453152027 + t * 1.061405429))))
        val res = poly * kotlin.math.exp(-x * x)
        return if (x >= 0) res else 2.0 - res
    }
    return erfc(kotlin.math.abs(z) / kotlin.math.sqrt(2.0))
}

// Format p-value similar to mizani's label_pvalue(add_p=True)
fun formatPValue(p: Double): String = when {
    p < 0.001 -> "p < 0.001"
    p < 0.01  -> "p < 0.01"
    p < 0.05  -> "p < 0.05"
    else      -> "p = ${ "%.2f".format(p) }"
}

fun starsFormatter(p: Double): String = when {
    p <= 0.001 -> "***"
    p <= 0.01  -> "**"
    p <= 0.05  -> "*"
    else       -> "ns"
}

In [5]:
val mpgDf = DataFrame.readCSV("https://raw.githubusercontent.com/JetBrains/lets-plot-docs/refs/heads/master/data/mpg.csv")
val mpg = mpgDf.toMap()
println("${mpgDf.rowsCount()} x ${mpgDf.columnsCount()}")
mpgDf.head()

234 x 12


untitled,manufacturer,model,displ,year,cyl,trans,drv,cty,hwy,fl,class
1,audi,a4,"1,800000",1999,4,auto(l5),f,18,29,p,compact
2,audi,a4,"1,800000",1999,4,manual(m5),f,21,29,p,compact
3,audi,a4,"2,000000",2008,4,manual(m6),f,20,31,p,compact
4,audi,a4,"2,000000",2008,4,auto(av),f,21,30,p,compact
5,audi,a4,"2,800000",1999,6,auto(l5),f,16,26,p,compact


## Adding Significance Bars (p-values) to Categorical Plots

#### Compute Significance Data

In [6]:
fun getPValuesData(
    df: DataFrame<*>,
    catCol: String,
    valCol: String,
    baseDy: Double,
    step: Double
): Map<String, List<Any?>> {
    val categories = df[catCol].distinct().toList().map { it.toString() }
    val pairs = categories.flatMapIndexed { i, a ->
        categories.drop(i + 1).map { b -> a to b }
    }
    val xminList = mutableListOf<String>()
    val xmaxList = mutableListOf<String>()
    val yList = mutableListOf<Double>()
    val pList = mutableListOf<Double>()
    pairs.forEachIndexed { i, (a, b) ->
        val x = df.filter { it[catCol].toString() == a }[valCol].toList().map { it.toString().toDouble() }
        val y = df.filter { it[catCol].toString() == b }[valCol].toList().map { it.toString().toDouble() }
        val p = mannWhitneyPValue(x, y)
        xminList += a
        xmaxList += b
        yList += baseDy + i * step
        pList += p
    }
    return mapOf("xmin" to xminList, "xmax" to xmaxList, "y" to yList, "p" to pList)
}

val pValuesData = getPValuesData(mpgDf, "drv", "hwy", 47.0, 4.0)
println("${pValuesData["xmin"]?.size} rows")
pValuesData

3 rows


{xmin=[f, f, 4], xmax=[4, r, r], y=[47.0, 51.0, 55.0], p=[1.3729654586124614E-27, 7.999377689136144E-11, 0.0425062896658749]}

In [7]:
val p = letsPlot(mpg) { x = "drv"; y = "hwy"; fill = "drv" } +
    geomBoxplot(alpha = .25) +
    geomJitter(height = 0.0, shape = 1, alpha = .25, showLegend = false, seed = 42) { color = "drv" }

#### Basic p-value Annotations

In [8]:
p + geomBracket(data = pValuesData) {  // <-- Pass p-values data to the brackets layer
    xmin = "xmin"; xmax = "xmax"; y = "y"; label = "p"
}

1.37297· 10 ​ −27 ​ 
 
 
 
 
 
 
 7.99938· 10 ​ −11 ​ 
 
 
 
 
 
 
 0.0425063 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 r 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 55 
 
 
 
 
 
 
 
 
 hwy 
 
 
 
 
 drv 
 
 
 
 
 
 
 
 
 drv 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 r

#### Format Labels with `labelFormat`

In [9]:
p + geomBracket(data = pValuesData, labelFormat = ".2~g") {
    xmin = "xmin"; xmax = "xmax"; y = "y"; label = "p"
}

1.4· 10 ​ −27 ​ 
 
 
 
 
 
 
 8· 10 ​ −11 ​ 
 
 
 
 
 
 
 0.043 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 r 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 55 
 
 
 
 
 
 
 
 
 hwy 
 
 
 
 
 drv 
 
 
 
 
 
 
 
 
 drv 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 r

#### Format Labels with `formatPValue()`

A helper that formats p-values as `p < 0.001`, `p < 0.01`, `p < 0.05`, or `p = X.XX`.

In [10]:
val pValuesDataLabeled = pValuesData + mapOf(
    "label" to (pValuesData["p"] as List<Double>).map { formatPValue(it) }
)

p + geomBracket(data = pValuesDataLabeled) {
    xmin = "xmin"; xmax = "xmax"; y = "y"; label = "label"
}

p < 0.001 
 
 
 
 
 
 
 p < 0.001 
 
 
 
 
 
 
 p < 0.05 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 r 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 55 
 
 
 
 
 
 
 
 
 hwy 
 
 
 
 
 drv 
 
 
 
 
 
 
 
 
 drv 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 r

#### Apply Custom Formatting Logic (Significance Stars)

Map p-values to custom strings (like `***` or `ns`) using your own function.

In [11]:
val pValuesDataStars = pValuesDataLabeled + mapOf(
    "star" to (pValuesData["p"] as List<Double>).map { starsFormatter(it) }
)

p + geomBracket(data = pValuesDataStars) {
    xmin = "xmin"; xmax = "xmax"; y = "y"; label = "star"
}

*** 
 
 
 
 
 
 
 *** 
 
 
 
 
 
 
 * 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 r 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 55 
 
 
 
 
 
 
 
 
 hwy 
 
 
 
 
 drv 
 
 
 
 
 
 
 
 
 drv 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 r

## Annotating Dodged Groups

To draw brackets between dodged groups, use `geomBracketDodge()`. Instead of using data coordinates, this layer uses group indices:

- `x`: The main category on the x-axis.
- `iStart`, `iEnd`: The zero-based indices of the dodged groups to compare.

In [12]:
// Group indices must be specified in the same order as they appear on the plot for other layers
val groupIndices = mapOf(4.0 to 0, 6.0 to 1, 8.0 to 2, 5.0 to 3)

val classes = mpgDf["class"].distinct().toList().map { it.toString() }

val xList = mutableListOf<String>()
val yList2 = mutableListOf<Double>()
val startList = mutableListOf<Int?>()
val endList = mutableListOf<Int?>()
val starList = mutableListOf<String>()

for (cls in classes) {
    val subDf = mpgDf.filter { it["class"].toString() == cls }
    val subData = getPValuesData(subDf, "cyl", "hwy", 47.0, 4.0)
    val xmins = subData["xmin"] as List<String>
    val xmaxs = subData["xmax"] as List<String>
    val ys = subData["y"] as List<Double>
    val ps = subData["p"] as List<Double>
    for (i in xmins.indices) {
        xList += cls
        yList2 += ys[i]
        startList += groupIndices[xmins[i].toDoubleOrNull()]
        endList += groupIndices[xmaxs[i].toDoubleOrNull()]
        starList += starsFormatter(ps[i])
    }
}

val pValuesGroupedData = mapOf(
    "x" to xList, "y" to yList2,
    "start" to startList, "end" to endList,
    "star" to starList
)
println("${xList.size} rows")
pValuesGroupedData

19 rows


{x=[compact, compact, compact, midsize, midsize, midsize, suv, suv, suv, minivan, pickup, pickup, pickup, subcompact, subcompact, subcompact, subcompact, subcompact, subcompact], y=[47.0, 51.0, 55.0, 47.0, 51.0, 55.0, 47.0, 51.0, 55.0, 47.0, 47.0, 51.0, 55.0, 47.0, 51.0, 55.0, 59.0, 63.0, 67.0], start=[0, 0, 1, 1, 1, 2, 2, 2, 1, 0, 1, 1, 2, 1, 1, 1, 2, 2, 0], end=[1, 3, 3, 2, 0, 0, 1, 0, 0, 1, 2, 0, 0, 2, 0, 3, 0, 3, 3], star=[***, ns, *, *, ***, *, **, ***, ***, ns, **, *, **, **, ***, *, ***, ns, ns]}

In [13]:
letsPlot(mpg) { x = "class"; y = "hwy"; fill = asDiscrete("cyl") } +
    geomBoxplot(alpha = .25) +
    geomPoint(
        position = positionJitterDodge(jitterWidth = .2, jitterHeight = 0.0, seed = 42),
        shape = 1, alpha = .25, showLegend = false
    ) { color = asDiscrete("cyl") } +
    geomBracketDodge(data = pValuesGroupedData) {
        x = "x"; y = "y"; iStart = "start"; iEnd = "end"; label = "star"
    } +
    scaleFillBrewer(palette = "Accent", format = "d") +
    scaleColorBrewer(palette = "Accent", format = "d") +
    ggsize(1000, 500)

*** 
 
 
 
 
 
 
 ns 
 
 
 
 
 
 
 * 
 
 
 
 
 
 
 * 
 
 
 
 
 
 
 *** 
 
 
 
 
 
 
 * 
 
 
 
 
 
 
 ** 
 
 
 
 
 
 
 *** 
 
 
 
 
 
 
 *** 
 
 
 
 
 
 
 ns 
 
 
 
 
 
 
 ** 
 
 
 
 
 
 
 * 
 
 
 
 
 
 
 ** 
 
 
 
 
 
 
 ** 
 
 
 
 
 
 
 *** 
 
 
 
 
 
 
 * 
 
 
 
 
 
 
 *** 
 
 
 
 
 
 
 ns 
 
 
 
 
 
 
 ns 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 compact 
 
 
 
 
 
 
 
 
 midsize 
 
 
 
 
 
 
 
 
 minivan 
 
 
 
 
 
 
 
 
 subcompact 
 
 
 
 
 
 
 
 
 suv 
 
 
 
 
 
 
 
 
 pickup 
 
 
 
 
 
 
 
 
 2seater 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 55 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 65 
 
 
 
 
 
 
 
 
 hwy 
 
 
 
 
 class 
 
 
 
 
 
 
 
 
 cyl 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 5

## Customizing Bracket Geometry

#### Bottom-Axis (Upside Down) Annotations

To place brackets at the bottom of a plot, use negative values for `lenStart` and `lenEnd`.  
You may also need to adjust `vjust` to position the labels correctly outside the bracket.

In [14]:
val upsideDownData = getPValuesData(mpgDf, "drv", "hwy", 8.0, -4.0).toMutableMap()
upsideDownData["label"] = (upsideDownData["p"] as List<Double>).map { formatPValue(it) }

println("${(upsideDownData["xmin"] as List<*>).size} rows")
upsideDownData

3 rows


{xmin=[f, f, 4], xmax=[4, r, r], y=[8.0, 4.0, 0.0], p=[1.3729654586124614E-27, 7.999377689136144E-11, 0.0425062896658749], label=[p < 0.001, p < 0.001, p < 0.05]}

In [15]:
p + geomBracket(
    data = upsideDownData,
    lenStart = -5, lenEnd = -5,  // Negative values to reverse the direction of the brackets
    vjust = 2.0                  // Specify vjust to move the labels under the brackets
) {
    xmin = "xmin"; xmax = "xmax"; y = "y"; label = "label"
}

p < 0.001 
 
 
 
 
 
 
 p < 0.001 
 
 
 
 
 
 
 p < 0.05 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 r 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 
 
 hwy 
 
 
 
 
 drv 
 
 
 
 
 
 
 
 
 drv 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 r

#### Precise Alignment with `tipLengthUnit = "identity"`

By default, bracket tips use a fixed length. Setting `tipLengthUnit = "identity"` allows you to specify tip lengths in **data units**.  
This is essential when you want a bracket tip to reach a specific coordinate, such as the exact top of a boxplot or a particular data point.

In [16]:
// Recall the y-coordinates from the datasets
val drvGroups = mpgDf.groupBy("drv")
val maxHwyByDrv = drvGroups.aggregate { maxOf("hwy") into "max_hwy" }.toMap()
println("Upper limits of boxes for each category: $maxHwyByDrv")
println("Dataset with p-values:")
pValuesDataLabeled

Upper limits of boxes for each category: {drv=[f, 4, r], max_hwy=[hwy, hwy, hwy]}
Dataset with p-values:


{xmin=[f, f, 4], xmax=[4, r, r], y=[47.0, 51.0, 55.0], p=[1.3729654586124614E-27, 7.999377689136144E-11, 0.0425062896658749], label=[p < 0.001, p < 0.001, p < 0.05]}

In [17]:
// The tip lengths are chosen so that each tip reaches either the previous bracket or the box
// Example (first bracket):
//   lenStart = y - upper_box_limit_at_xmin - 1
//     -> 47 - 44 - 1 = 2
//   lenEnd   = y - upper_box_limit_at_xmax - 1
//     -> 47 - 28 - 1 = 18
p + geomBracket(
    data = pValuesDataLabeled,
    tipLengthUnit = "identity"  // identity units (data space)
) {
    xmin = "xmin"; xmax = "xmax"; y = "y"; label = "label"
    lenStart = listOf(2, 1, 1)
    lenEnd = listOf(18, 24, 1)
}

p < 0.001 
 
 
 
 
 
 
 p < 0.001 
 
 
 
 
 
 
 p < 0.05 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 r 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 55 
 
 
 
 
 
 
 
 
 hwy 
 
 
 
 
 drv 
 
 
 
 
 
 
 
 
 drv 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 r

## Non-Statistical Annotations: Range Grouping

Brackets can be used for more than just drawing p-values.  
For example, they can be used to highlight cluster boundaries in the following scatter plot:

In [18]:
fun getContinuousBracketsData(
    df: DataFrame<*>,
    catCol: String,
    primaryValCol: String,
    secondaryValCol: String,
    stepMult: Double
): Map<String, List<Any?>> {
    val secondaryVals = df[secondaryValCol].toList().map { it.toString().toDouble() }
    val secMin = secondaryVals.min()
    val secMax = secondaryVals.max()
    val step = (secMax - secMin) * stepMult
    val base = secMax + step

    val categoryOrder = listOf("r", "4", "f")
    val categories = categoryOrder.filter { cat -> df[catCol].toList().any { it.toString() == cat } }

    val catList = mutableListOf<String>()
    val minPrimary = mutableListOf<Double>()
    val maxPrimary = mutableListOf<Double>()
    val secondaryLevel = mutableListOf<Double>()

    categories.forEachIndexed { idx, cat ->
        val vals = df.filter { it[catCol].toString() == cat }[primaryValCol]
            .toList().map { it.toString().toDouble() }
        catList += cat
        minPrimary += vals.min()
        maxPrimary += vals.max()
        secondaryLevel += base + idx * step
    }

    return mapOf(
        catCol to catList,
        "min_$primaryValCol" to minPrimary,
        "max_$primaryValCol" to maxPrimary,
        "${secondaryValCol}_level" to secondaryLevel
    )
}

val hwyBracketsData = getContinuousBracketsData(mpgDf, "drv", "hwy", "cty", 0.1)
val ctyBracketsData = getContinuousBracketsData(mpgDf, "drv", "cty", "hwy", 0.05)

letsPlot() +
    geomPoint(data = mpg, alpha = .25, showLegend = false) { x = "hwy"; y = "cty"; color = "drv" } +
    // Horizontal brackets for hwy ranges
    geomBracket(data = hwyBracketsData) {
        xmin = "min_hwy"; xmax = "max_hwy"; y = "cty_level"; label = "drv"; color = "drv"
    } +
    // Vertical brackets for cty ranges
    geomBracket(data = ctyBracketsData) {
        x = "hwy_level"; ymin = "min_cty"; ymax = "max_cty"; label = "drv"; color = "drv"
    } +
    themeMinimal() +
    labs(x = "hwy", y = "cty")

r 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 r 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 f 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 
 
 cty 
 
 
 
 
 hwy